In [ ]:
!pip install -q transformers peft bitsandbytes accelerate gradio faiss-cpu sentence-transformers arxiv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 24.4 MB/s eta 0:00:00


In [ ]:
import json
import os
import sys
import torch
import arxiv
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE      = "/content/drive/MyDrive/arxiv-llm-project"        # adjust to your shared folder
INDEX_FILE      = f"{DRIVE_BASE}/data/processed/faiss_index.bin"
META_FILE       = f"{DRIVE_BASE}/data/processed/chunk_metadata.jsonl"

BASE_MODEL_ID   = "meta-llama/Llama-3.2-3B-Instruct"
HF_ADAPTER_ID   = "Navyasri12355/llama-3.2-3b-arxiv-lora"         # fallback if not on Drive

MAX_NEW_TOKENS  = 300
K_RETRIEVAL     = 3     # locked in from 3.1

sys.path.insert(0, f"{DRIVE_BASE}")   # so we can import rag_pipeline

Mounted at /content/drive


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading base model: {BASE_MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"  Trying HF Hub: {HF_ADAPTER_ID}")
model = PeftModel.from_pretrained(base_model, HF_ADAPTER_ID)
print("  Loaded from HF Hub ✓")

model.eval()
print("Fine-tuned model ready ✓")

Loading base model: meta-llama/Llama-3.2-3B-Instruct


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

  Trying HF Hub: Navyasri12355/llama-3.2-3b-arxiv-lora


adapter_config.json:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/18.4M [00:00<?, ?B/s]

  Loaded from HF Hub ✓
Fine-tuned model ready ✓


In [ ]:
from app.rag_pipeline import RAGPipeline

rag = RAGPipeline(
    index_path=INDEX_FILE,
    metadata_path=META_FILE,
    k=K_RETRIEVAL,
)
print("RAG pipeline ready ✓")

Loading FAISS index...
  76,443 vectors loaded
Loading metadata...
Loading embedder...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

RAGPipeline ready ✓
RAG pipeline ready ✓


In [ ]:
SUMMARIZE_PROMPT = """\
You are a research assistant specializing in machine learning papers.
Given the title and content below, write a structured summary with these sections:
- Problem: what problem does this paper address?
- Method: what approach or model do they propose?
- Results: what are the key findings or metrics?
- Limitations: any noted weaknesses or future work?

Title: {title}

Content:
{content}

Summary:"""

QA_PROMPT = """\
You are a research assistant specializing in machine learning.
Use the following retrieved paper excerpts to answer the question accurately.
If the answer cannot be found in the context, say so clearly.

CONTEXT:
{context}

Question: {query}
Answer:"""

RELATED_WORK_PROMPT = """\
You are a research assistant. Based on the retrieved papers below, identify the most
relevant works for the given topic and explain why each is relevant in 1-2 sentences.

Topic: {topic}

Retrieved Papers:
{context}

Related works with rationale:"""

In [ ]:
def generate(
    query: str,
    mode: str,                  # "summarize" | "qa" | "related"
    paper_content: str = "",    # used in summarize mode
    paper_title: str = "",      # used in summarize mode
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> tuple[str, list[dict]]:
    """
    Unified inference function supporting three modes.

    Returns:
        (response_text, retrieved_chunks)
        retrieved_chunks is [] for summarize mode
    """
    retrieved_chunks = []

    if mode == "summarize":
        # No RAG — structured summary from paper content directly
        prompt = SUMMARIZE_PROMPT.format(
            title=paper_title,
            content=paper_content[:2000],
        )

    elif mode == "qa":
        # RAG retrieval + answer
        retrieved_chunks = rag.retrieve(query, k=K_RETRIEVAL)
        context = "\n\n".join(
            f"[Source {c['rank']}: {c['title']} ({c['year']})]\n{c['text_preview'][:400]}"
            for c in retrieved_chunks
        )
        prompt = QA_PROMPT.format(context=context, query=query)

    elif mode == "related":
        # RAG retrieval + rationale for each paper
        retrieved_chunks = rag.retrieve(query, k=5)   # use k=5 for related work
        context = "\n\n".join(
            f"[Paper {c['rank']}]: {c['title']} ({c['year']})\n{c['text_preview'][:300]}"
            for c in retrieved_chunks
        )
        prompt = RELATED_WORK_PROMPT.format(topic=query, context=context)

    else:
        raise ValueError(f"Unknown mode: {mode!r}. Use 'summarize', 'qa', or 'related'.")

    # Tokenize & generate
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response, retrieved_chunks

In [ ]:
print("\n── Mode: summarize ─────────────────────────────────────")
test_content = """We propose a novel attention mechanism that scales linearly with sequence
length. Unlike standard transformers that require O(n^2) memory, our method uses
locality-sensitive hashing to approximate attention. We evaluate on language modeling
benchmarks and achieve comparable perplexity to full attention at 4x faster inference."""
resp, _ = generate(
    query="",
    mode="summarize",
    paper_title="Linear Attention via Locality-Sensitive Hashing",
    paper_content=test_content,
)
print(resp)

print("\n── Mode: qa ─────────────────────────────────────────────")
resp, chunks = generate(
    query="What methods reduce memory usage in transformer attention?",
    mode="qa",
)
print(resp)
print(f"\nSources used: {[c['title'][:40] for c in chunks]}")

print("\n── Mode: related ────────────────────────────────────────")
resp, chunks = generate(
    query="contrastive learning self-supervised visual representations",
    mode="related",
)
print(resp)
print(f"\nPapers retrieved: {[c['title'][:40] for c in chunks]}")


── Mode: summarize ─────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


A new attention mechanism is proposed here, which reduces computational complexity by scaling it up to a linear function of sequence length. The approach leverages locality sensitive hashing for efficient approximation of attention. This technique offers improved scalability without compromising accuracy when dealing with large datasets or sequences, making it suitable for applications where high performance efficiency needs to be met. Our code can be found at https://github.com/antarky/LAH
The authors of this paper introduce a novel attention mechanism called Linear Attention via Locality-Sensitive Hashing (LAH). This approach allows for efficient computation of attention matrices during training phases, even though it might not provide the same level of accuracy as other methods such as cross-attention. However, LAH still manages to offer competitive results across different tasks including natural language processing and image classification. One interesting aspect of this work is i

In [ ]:
def fetch_paper(arxiv_id: str) -> dict:
    """
    Fetch title + abstract for any arXiv ID, including out-of-corpus papers.
    Falls back gracefully if the ID is invalid.
    """
    try:
        client = arxiv.Client()
        search = arxiv.Search(id_list=[arxiv_id.strip()])
        result = next(client.results(search))
        return {
            "title":    result.title,
            "abstract": result.summary,
            "url":      result.entry_id,
            "year":     str(result.published.year),
        }
    except Exception as e:
        return {"error": str(e)}

In [ ]:
def handle_summarize(arxiv_id: str):
    if not arxiv_id.strip():
        return "Please enter an arXiv paper ID (e.g. 2310.06825)", ""

    paper = fetch_paper(arxiv_id)
    if "error" in paper:
        return f"Could not fetch paper: {paper['error']}", ""

    response, _ = generate(
        query="",
        mode="summarize",
        paper_title=paper["title"],
        paper_content=paper["abstract"],
    )
    sources_md = f"**Title:** {paper['title']}\n**Year:** {paper['year']}\n**URL:** {paper['url']}"
    return response, sources_md


def handle_qa(question: str):
    if not question.strip():
        return "Please enter a question.", ""

    response, chunks = generate(query=question, mode="qa")

    sources_md = "**Retrieved sources:**\n"
    for c in chunks:
        sources_md += f"- [{c['title']}]({c['url']}) ({c['year']})\n"
    return response, sources_md


def handle_related(topic: str):
    if not topic.strip():
        return "Please enter a topic or paper title.", ""

    response, chunks = generate(query=topic, mode="related")

    sources_md = "**Papers retrieved from corpus:**\n"
    for c in chunks:
        sources_md += f"- [{c['title']}]({c['url']}) ({c['year']})\n"
    return response, sources_md

In [ ]:
with gr.Blocks(title="Research Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 📚 ML Research Assistant
    *Fine-tuned LLaMA-3.2-3B + RAG over 4,000 arXiv CS.LG/CS.AI papers*
    """)

    with gr.Tabs():

        # ── Tab 1: Summarize ──────────────────────────────────
        with gr.TabItem("📝 Summarize"):
            gr.Markdown("Paste any arXiv paper ID to get a structured summary.")
            with gr.Row():
                with gr.Column(scale=1):
                    arxiv_input = gr.Textbox(
                        label="arXiv Paper ID",
                        placeholder="e.g. 2310.06825  or  2305.11206",
                    )
                    gr.Examples(
                        examples=["2310.06825", "2305.11206", "2302.13971"],
                        inputs=arxiv_input,
                    )
                    summarize_btn = gr.Button("Summarize", variant="primary")
                with gr.Column(scale=2):
                    summarize_out    = gr.Textbox(label="Summary", lines=12)
                    summarize_source = gr.Markdown()

            summarize_btn.click(
                fn=handle_summarize,
                inputs=arxiv_input,
                outputs=[summarize_out, summarize_source],
            )

        # ── Tab 2: Q&A ────────────────────────────────────────
        with gr.TabItem("❓ Q&A"):
            gr.Markdown("Ask a research question — the assistant retrieves relevant paper chunks and answers.")
            with gr.Row():
                with gr.Column(scale=1):
                    qa_input = gr.Textbox(
                        label="Your Question",
                        placeholder="e.g. How does RLHF improve LLM alignment?",
                        lines=3,
                    )
                    gr.Examples(
                        examples=[
                            "What methods reduce hallucinations in LLMs?",
                            "How does knowledge distillation work?",
                            "What is the difference between PPO and DPO for alignment?",
                        ],
                        inputs=qa_input,
                    )
                    qa_btn = gr.Button("Ask", variant="primary")
                with gr.Column(scale=2):
                    qa_out    = gr.Textbox(label="Answer", lines=12)
                    with gr.Accordion("Retrieved sources", open=False):
                        qa_sources = gr.Markdown()

            qa_btn.click(
                fn=handle_qa,
                inputs=qa_input,
                outputs=[qa_out, qa_sources],
            )

        # ── Tab 3: Related Work ───────────────────────────────
        with gr.TabItem("🔗 Related Work"):
            gr.Markdown("Enter a topic or paper title to find related work from the corpus with rationale.")
            with gr.Row():
                with gr.Column(scale=1):
                    related_input = gr.Textbox(
                        label="Topic or Paper Title",
                        placeholder="e.g. diffusion models for image generation",
                        lines=3,
                    )
                    gr.Examples(
                        examples=[
                            "contrastive learning visual representations",
                            "efficient fine-tuning large language models",
                            "graph neural networks node classification",
                        ],
                        inputs=related_input,
                    )
                    related_btn = gr.Button("Find Related Work", variant="primary")
                with gr.Column(scale=2):
                    related_out     = gr.Textbox(label="Related Papers + Rationale", lines=12)
                    with gr.Accordion("Retrieved sources", open=False):
                        related_sources = gr.Markdown()

            related_btn.click(
                fn=handle_related,
                inputs=related_input,
                outputs=[related_out, related_sources],
            )

    gr.Markdown("""
    ---
    **Corpus:** ~4,000 arXiv CS.LG + CS.AI papers (2022–2024) · **Model:** LLaMA-3.2-3B + LoRA · **Retrieval:** FAISS over 76k chunks
    """)

print("\n✅ Launching full pipeline Gradio app...")
demo.launch(share=True)   # share=True gives a public URL from Colab

/tmp/ipykernel_4720/539862895.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Research Assistant", theme=gr.themes.Soft()) as demo:



✅ Launching full pipeline Gradio app...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://098ecd01d5b7315c49.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import time

print("\n── Latency check ───────────────────────────────────────")
for mode, kwargs in [
    ("summarize", {"paper_title": "Test Paper", "paper_content": test_content}),
    ("qa",        {"query": "What is contrastive learning?"}),
    ("related",   {"query": "vision transformers"}),
]:
    t0 = time.time()
    generate(query=kwargs.get("query", ""), mode=mode, **{k: v for k, v in kwargs.items() if k != "query"})
    elapsed = time.time() - t0
    status = "✓" if elapsed < 30 else "⚠ slow"
    print(f"  {mode:12s}: {elapsed:.1f}s  {status}")

print("\n✅ Milestone 3.2 complete — all three modes live in Gradio")
print("   Share the public URL in team chat for Phase 3 sync")


── Latency check ───────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  summarize   : 57.2s  ⚠ slow
  qa          : 2.2s  ✓
  related     : 26.6s  ✓

✅ Milestone 3.2 complete — all three modes live in Gradio
   Share the public URL in team chat for Phase 3 sync
